# CortiPy — Surgical View of the Signal & Metadata Layers

**Companion notebook to** *"CortiPy: Unifying Real-Time Acquisition with Semantic Columnar Storage for Evoked Potentials"* (EMBC).

This notebook is a **figure factory and Q&A crib sheet**. It dissects, against *real generated files*, the two
design choices the paper argues for:

1. **Signal layer — Apache Parquet vs EDF.** Columnar table (footer + row groups + column chunks + per-column
   statistics) vs row-record container (fixed header + interleaved data records of 16-bit integers).
2. **Metadata layer — BIDS sidecars vs SBIDS JSON-LD.** Meaning scattered across a folder/filename convention
   vs one queryable knowledge graph linked to the raw file.

Every figure is exported to `presentation_artifacts/figures/` as **PDF + SVG (vector, editable text) and PNG @300/@600 dpi**
so it drops straight into slides.

**How to run:** open this notebook from the repo root and *Run All*. It **creates all data and directories itself**
(nothing to download) on randomly generated EEG. Figures render inline and are written to `presentation_artifacts/figures/`.
The **final cell tidies up** the generated folder and leaves a short `README.md` — set `CLEAN_ARTIFACTS = False`
in the setup cell to keep the exported image files.

> Data is 100% synthetic. The headline numbers from the paper (Apple M2 Max, v0.1.2) are quoted where relevant;
> the live cells reproduce the *mechanism* and the *scaling behaviour*, not the absolute hardware timings.


In [ ]:
# === Setup: imports, palette, publication rcParams, figure toolkit, multi-format export ======
import os, sys, json, math, time, struct, textwrap, hashlib, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
from matplotlib.lines import Line2D

HERE = Path.cwd().resolve()
if HERE.name == "presentation_artifacts" and (HERE / "EMBC_CortiPy_public_demo.ipynb").exists():
    NOTEBOOK_DIR = HERE
    REPO_ROOT = HERE.parent
elif (HERE / "presentation_artifacts" / "EMBC_CortiPy_public_demo.ipynb").exists():
    NOTEBOOK_DIR = HERE / "presentation_artifacts"
    REPO_ROOT = HERE
else:
    NOTEBOOK_DIR = HERE
    REPO_ROOT = HERE.parent if HERE.name == "presentation_artifacts" else HERE
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import mne
import pyarrow as pa
import pyarrow.parquet as pq
import pyedflib

from cortipy.shared.dataset import CortiDataset
from cortipy.shared.bids import BIDSLoader
from cortipy.shared.sbids import SBIDSLoader, to_sbids

try:
    from IPython.display import display
except Exception:  # plain-Python fallback so the notebook source also runs as a script
    def display(*objs, **_):
        for o in objs:
            print(o)

mne.set_log_level("ERROR")

# --- palette (consistent across every figure; maps to the paper's framing) ------------------
PAL = dict(
    ink="#16222b", muted="#5b6b75", line="#9fb0ba", grid="#e3e9ec",
    panel="#ffffff", edge="#c2cdd5",
    parquet="#1f7a5a", parquet_bg="#e6f3ed",
    edf="#bf7d12", edf_bg="#fbf1da",
    meta="#2b6cb0", meta_bg="#e7f1fb",
    bids="#5a4b8a", bids_bg="#ece8f6",
    sbids="#a32c52", sbids_bg="#fbe9ef",
)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "DejaVu Sans",
    "font.size": 15,
    "axes.titlesize": 21,
    "axes.titleweight": "bold",
    "axes.labelsize": 16,
    "axes.edgecolor": PAL["muted"],
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "svg.fonttype": "none",   # keep SVG text as editable text
    "pdf.fonttype": 42,        # embed TrueType so PDF text stays editable
    "ps.fonttype": 42,
})

SLIDE = (13.33, 7.5)  # 16:9
ARTIFACT_DIR = NOTEBOOK_DIR   # this notebook lives in presentation_artifacts; outputs are generated here
FIG_DIR = ARTIFACT_DIR / "figures"
ARTIFACT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
for _f in FIG_DIR.glob("*"):          # fresh start so the dir never accumulates stale figures
    if _f.is_file():
        _f.unlink()

# --- run config -----------------------------------------------------------------------------
# The LAST cell tidies up generated files. Set False (or env CORTIPY_CLEAN=false) to keep the figures.
CLEAN_ARTIFACTS = os.environ.get("CORTIPY_CLEAN", "true").lower() not in ("0", "false", "no")
SWEEP_DURS = [0.5, 2.0, 8.0, 30.0, 120.0, 240.0]   # seconds @ 64 ch — signal read-scaling sweep
META_NS = [1, 4, 16, 32]                            # cohort sizes (N) — metadata-scaling sweep

def save_fig(fig, name):
    """Export one figure as vector PDF+SVG and raster PNG@300/@600 for slide decks."""
    fig.savefig(FIG_DIR / f"{name}.pdf")
    fig.savefig(FIG_DIR / f"{name}.svg")
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300)
    fig.savefig(FIG_DIR / f"{name}@2x.png", dpi=600)
    print(f"  saved figures/{name}.(pdf|svg|png@300|@2x.png)")
    return name

# --- tiny diagram toolkit (square coords so corners/edges are undistorted) --------------------
def canvas(title=None, subtitle=None, w=13.33, h=7.5):
    fig, ax = plt.subplots(figsize=(w, h))
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100 * h / w)
    ax.set_aspect("equal")
    ax.axis("off")
    top = 100 * h / w
    if title:
        ax.text(2.5, top - 2.2, title, fontsize=23, fontweight="bold", color=PAL["ink"], va="top")
    if subtitle:
        ax.text(2.5, top - 6.0, subtitle, fontsize=14.5, color=PAL["muted"], va="top")
    return fig, ax, top

def rect(ax, x, y, w, h, fc, ec=None, lw=2.0, z=2, alpha=1.0):
    r = Rectangle((x, y), w, h, facecolor=fc, edgecolor=ec or PAL["edge"],
                  linewidth=lw, zorder=z, alpha=alpha)
    ax.add_patch(r)
    return r

def label(ax, x, y, s, size=14, color=None, weight="normal", ha="left", va="center", family=None, z=6):
    ax.text(x, y, s, fontsize=size, color=color or PAL["ink"], fontweight=weight,
            ha=ha, va=va, family=family, zorder=z)

def arrow(ax, p1, p2, color=None, lw=2.4, style="-|>", rad=0.0, z=5, ls="-"):
    a = FancyArrowPatch(p1, p2, arrowstyle=style, mutation_scale=16, lw=lw, ls=ls,
                        color=color or PAL["ink"], zorder=z,
                        connectionstyle=f"arc3,rad={rad}")
    ax.add_patch(a)
    return a

def show_table(df, caption=None):
    sty = (df.style.hide(axis="index")
             .set_table_styles([
                 {"selector": "th", "props": [("font-size", "15px"), ("text-align", "left"),
                                              ("background", "#f3f6f8"), ("padding", "6px 10px")]},
                 {"selector": "td", "props": [("font-size", "14px"), ("padding", "5px 10px"),
                                              ("border-bottom", "1px solid #e6ebee")]},
             ]))
    if caption:
        sty = sty.set_caption(caption).set_table_styles(
            [{"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "700"),
                                               ("text-align", "left"), ("padding", "4px 0 8px")]}],
            overwrite=False)
    display(sty)

def median_ms(fn, n=9):
    """Median wall-clock of fn() in ms, after one warm-up call."""
    fn()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts))

print("Setup ready.")
print(f"  python={'.'.join(map(str, __import__('sys').version_info[:3]))}  mne={mne.__version__}  "
      f"pyarrow={pa.__version__}  pyedflib={pyedflib.__version__}")
print(f"  figures -> {FIG_DIR.resolve()}")


## Part A — One logical recording, several physical containers

We synthesize a single deterministic EEG recording, then write it out through CortiPy's real I/O paths:
BIDS+Parquet, BIDS+EDF, and SBIDS (JSON-LD + Parquet). Everything downstream dissects *these* files.


In [ ]:
# === Build one recording and materialize every container (real CortiPy I/O) ==================
CH = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4"]
FS = 250.0
DUR = 30.0  # seconds -> 7500 samples; large enough for honest byte-level layout numbers

dataset = CortiDataset.generate_eeg_samples(
    sampling_rate=FS, duration_s=DUR, channel_names=CH, channel_count=len(CH),
    seed=2026, dataset_name="embc_demo",
)
N_SAMPLES, N_CH = dataset.data.shape

bids_common = dict(subject="01", session="demo", task="formatdemo", run="01", overwrite=True,
                   dataset_description={"Name": "CortiPy EMBC surgical demo"})
parquet_root = ARTIFACT_DIR / "bids_parquet"
edf_root = ARTIFACT_DIR / "bids_edf"

parquet_path = dataset.to_bids(parquet_root, format="parquet", **bids_common)
edf_path = dataset.to_bids(edf_root, format="edf", **bids_common)
sbids_path = dataset.to_sbids(ARTIFACT_DIR / "sbids_meta_demo.jsonld", export_format="parquet")

# fidelity: parquet is float64 (lossless), EDF is 16-bit (lossy) — measured later, surgically
preview = pd.DataFrame(dataset.data[:8], columns=dataset.raw.ch_names).round(3)
print(f"recording: {N_SAMPLES} samples x {N_CH} channels @ {FS:.0f} Hz  ({DUR:.0f} s)")
print(f"  BIDS+Parquet : {parquet_path.relative_to(ARTIFACT_DIR)}  ({parquet_path.stat().st_size/1024:.1f} KB)")
print(f"  BIDS+EDF     : {edf_path.relative_to(ARTIFACT_DIR)}  ({edf_path.stat().st_size/1024:.1f} KB)")
print(f"  SBIDS        : {sbids_path.name}  + raw_data/{sbids_path.stem}.parquet")
show_table(preview, caption="First 8 samples of the synthetic recording (the bytes we trace below)")
# === Figure 1 — logical model: physical container changes, analysis object does not ==========
fig, ax, top = canvas(
    "One recording object, several physical containers",
    "CortiPy keeps the analysis object (MNE RawArray) stable while the on-disk bytes change.")

# central logical object
cx, cy, cw, chgt = 36, 26, 28, 12
rect(ax, cx, cy, cw, chgt, PAL["panel"], PAL["ink"], lw=2.6)
label(ax, cx + cw/2, cy + chgt - 3.2, "EEG recording (logical)", size=16, weight="bold", ha="center")
label(ax, cx + cw/2, cy + chgt/2 - 0.3, "MNE RawArray", size=14, ha="center", color=PAL["muted"])
label(ax, cx + cw/2, cy + 2.6, f"{N_SAMPLES} samples x {N_CH} ch @ {FS:.0f} Hz", size=12.5, ha="center", color=PAL["muted"])

def container(x, y, w, h, title, lines, cfill, cedge):
    rect(ax, x, y, w, h, cfill, cedge, lw=2.4)
    label(ax, x + 2.2, y + h - 2.8, title, size=14, weight="bold", color=cedge)
    for i, t in enumerate(lines):
        label(ax, x + 2.2, y + h - 6.0 - i*2.5, t, size=11, color=PAL["ink"])

container(74, 34, 24, 12, "Parquet  (signal)",
          ["columnar table", "footer + statistics", "projection pushdown"], PAL["parquet_bg"], PAL["parquet"])
container(74, 20, 24, 12, "EDF  (signal)",
          ["fixed header", "interleaved records", "16-bit integers"], PAL["edf_bg"], PAL["edf"])
container(2, 34, 24, 12, "BIDS  (metadata)",
          ["folder + filename", "scattered sidecars", ".json / .tsv"], PAL["bids_bg"], PAL["bids"])
container(2, 20, 24, 12, "SBIDS  (metadata)",
          ["one JSON-LD graph", "typed nodes + links", "contentUrl -> raw"], PAL["sbids_bg"], PAL["sbids"])

arrow(ax, (cx + cw, cy + chgt*0.68), (74, 40), PAL["parquet"], rad=-0.16)
arrow(ax, (cx + cw, cy + chgt*0.30), (74, 26), PAL["edf"], rad=0.16)
arrow(ax, (cx, cy + chgt*0.68), (26, 40), PAL["bids"], rad=0.16)
arrow(ax, (cx, cy + chgt*0.30), (26, 26), PAL["sbids"], rad=-0.16)

label(ax, 50, 10, "Same recording, different bytes.   Part B: how is the signal stored?   "
                  "Part C: how is its meaning stored?", size=13, ha="center", color=PAL["muted"])
save_fig(fig, "fig1_logical_model")
plt.show(); plt.close(fig)


## Part B — Surgical: the signal layer (Parquet vs EDF)

Both files hold the *same* numbers. The difference is **organization on disk**, and that organization is what
makes a single-channel read cheap or expensive.


In [ ]:
# === Parquet anatomy: read the real footer (schema, row groups, per-column stats & offsets) ==
md = pq.read_metadata(parquet_path)   # reads the footer
rg0 = md.row_group(0)
PQ = {"created_by": md.created_by, "version": md.format_version, "nrows": md.num_rows,
      "ncols": md.num_columns, "nrg": md.num_row_groups, "footer": md.serialized_size}

rows = []
for i in range(PQ["ncols"]):
    c = rg0.column(i)
    st = c.statistics
    rows.append({
        "column": c.path_in_schema,
        "phys. type": c.physical_type,
        "codec": c.compression,
        "comp. bytes": c.total_compressed_size,
        "uncomp. bytes": c.total_uncompressed_size,
        "min (uV)": round(st.min, 2) if (st and st.has_min_max) else None,
        "max (uV)": round(st.max, 2) if (st and st.has_min_max) else None,
        "data offset": c.data_page_offset,
    })
parquet_cols = pd.DataFrame(rows)
del c, st, rg0, md   # drop ALL reader/column refs so the footer mmap is released (Windows file lock)

print(f"created_by      : {PQ['created_by']}")
print(f"format version  : {PQ['version']}")
print(f"rows            : {PQ['nrows']:,}   columns: {PQ['ncols']}   row groups: {PQ['nrg']}")
print(f"footer size     : {PQ['footer']:,} bytes  (thrift-encoded FileMetaData at end of file)")
fp1 = parquet_cols.iloc[0]
print(f"\nRead ONE channel ({fp1['column']}): seek to byte {int(fp1['data offset']):,}, "
      f"read {int(fp1['comp. bytes']):,} bytes -> done. The other {PQ['ncols']-1} columns are never touched.")
show_table(parquet_cols,
           caption="Parquet column chunks in row group 0 — every channel is a typed, independently addressable column")
# === Figure 2 — Parquet physical layout (block widths proportional to real compressed bytes) =
fig, ax, top = canvas(
    "Parquet file anatomy — columnar, footer-indexed",
    "To read one channel: consult the footer, seek to that column chunk, read only its bytes (projection pushdown).")

x0, x1 = 4.0, 96.0
yb, h = 24.0, 14.0
magic = 3.0
foot = 16.0
body_x = x0 + magic
body_w = (x1 - x0) - 2*magic - foot

comp = parquet_cols["comp. bytes"].to_numpy(dtype=float)
frac = comp / comp.sum()
widths = frac * body_w

# magic + footer
rect(ax, x0, yb, magic, h, "#eef2f4", PAL["muted"]); label(ax, x0+magic/2, yb+h/2, "PAR1", size=9, ha="center", va="center", color=PAL["muted"])
rect(ax, x1-magic, yb, magic, h, "#eef2f4", PAL["muted"]); label(ax, x1-magic/2, yb+h/2, "PAR1", size=9, ha="center", va="center", color=PAL["muted"])
# row group bracket
rect(ax, body_x, yb, body_w, h, PAL["parquet_bg"], PAL["parquet"], lw=2.2, alpha=0.45)
label(ax, body_x, yb+h+2.2, f"Row group 0  ({PQ['nrows']:,} rows)", size=12.5, weight="bold", color=PAL["parquet"])

cx = body_x
for i, w in enumerate(widths):
    hot = (i == 0)
    rect(ax, cx, yb+1.4, max(w-0.6, 1.2), h-2.8,
         "#6fb89a" if hot else PAL["parquet_bg"], PAL["parquet"], lw=2.6 if hot else 1.6, z=4 if hot else 3)
    label(ax, cx + w/2, yb + h - 3.2, parquet_cols['column'][i], size=10.5, ha="center", weight="bold",
          color="white" if hot else PAL["parquet"], z=6)
    label(ax, cx + w/2, yb + 3.0, f"{int(comp[i])}B", size=8.5, ha="center",
          color="white" if hot else PAL["muted"], z=6)
    cx += w
label(ax, body_x + body_w/2, yb - 2.6, "one column chunk per channel (contiguous on disk)", size=11, ha="center", color=PAL["muted"])

# footer
rect(ax, x1-magic-foot, yb, foot, h, PAL["meta_bg"], PAL["meta"], lw=2.2)
label(ax, x1-magic-foot/2, yb+h-3.0, "Footer", size=12, ha="center", weight="bold", color=PAL["meta"])
for j, t in enumerate(["schema", "min / max stats", "byte offsets"]):
    label(ax, x1-magic-foot/2, yb+h-6.8-j*2.6, t, size=9.5, ha="center", color=PAL["meta"])

# projection read annotation: highlight the target column + a short callout (no crossing arrow)
target_i = 0
tx = body_x + widths[:target_i].sum() + widths[target_i]/2
arrow(ax, (tx, yb-3.6), (tx, yb+0.6), PAL["ink"], lw=2.4)
label(ax, tx, yb-5.2, "read only\nthis column", size=10.5, ha="center", va="top", color=PAL["ink"])
label(ax, 50, top-9.5,
      f'Query "{parquet_cols["column"][target_i]}":  footer -> byte offset {int(parquet_cols["data offset"][target_i]):,} '
      f'-> read {int(comp[target_i]):,} B  (1 of {PQ["ncols"]} columns)', size=13, ha="center", color=PAL["ink"])
label(ax, 50, 7.5, "Footer statistics also let a reader skip whole row groups whose [min, max] cannot match a filter.",
      size=11.5, ha="center", color=PAL["muted"])
save_fig(fig, "fig2_parquet_layout")
plt.show(); plt.close(fig)


In [ ]:
# === EDF anatomy: parse the real ASCII header bytes + per-signal headers + quantization =======
raw_hdr = Path(edf_path).read_bytes()[:256]
def _f(a, b):
    return raw_hdr[a:b].decode("ascii", "replace").strip()

edf_header = {
    "version": repr(_f(0, 8)),
    "local patient": _f(8, 88) or "(empty)",
    "local recording": _f(88, 168) or "(empty)",
    "start date": _f(168, 176),
    "start time": _f(176, 184),
    "bytes in header": _f(184, 192),
    "reserved": _f(192, 236),
    "data records": _f(236, 244),
    "record duration (s)": _f(244, 252),
    "signals (ns)": _f(252, 256),
}
print("EDF main header record (first 256 bytes, ASCII):")
for k, v in edf_header.items():
    print(f"  {k:22s}: {v}")

r = pyedflib.EdfReader(str(edf_path))
try:
    ns = r.signals_in_file
    labels = r.getSignalLabels()
    drec_dur = float(r.datarecord_duration)
    n_drec = int(r.datarecords_in_file)
    rows = []
    rt_err = []
    for i in range(ns):
        pmin, pmax = r.getPhysicalMinimum(i), r.getPhysicalMaximum(i)
        dmin, dmax = r.getDigitalMinimum(i), r.getDigitalMaximum(i)
        sfreq_i = r.getSampleFrequency(i)
        spr = int(round(sfreq_i * drec_dur))
        step = (pmax - pmin) / (dmax - dmin)  # quantization step (uV per least-significant bit)
        sig = r.readSignal(i)
        rt_err.append(float(np.max(np.abs(sig - dataset.data[:len(sig), i]))))
        rows.append({
            "signal": labels[i],
            "phys min/max (uV)": f"{pmin:.1f} / {pmax:.1f}",
            "dig min/max": f"{int(dmin)} / {int(dmax)}",
            "samples/record": spr,
            "quant. step (uV)": round(step, 4),
        })
    edf_sig = pd.DataFrame(rows)
finally:
    r.close()

# Byte budget read straight from the real header + file size (most surgical & self-consistent).
ns_total = int(edf_header["signals (ns)"])        # 8 EEG + 1 mandatory EDF+ annotation signal
header_bytes = int(edf_header["bytes in header"])  # == 256 * (ns_total + 1)
n_records = int(edf_header["data records"])
total_bytes = Path(edf_path).stat().st_size
record_bytes = (total_bytes - header_bytes) // n_records
edf_max_err = float(np.max(rt_err))

# Parquet round-trip is exact (float64)
pq_back = pd.read_parquet(parquet_path).to_numpy()
pq_max_err = float(np.max(np.abs(pq_back - dataset.data)))

print(f"\nsignals     : {ns} EEG data signals + {ns_total - ns} EDF+ annotation signal -> ns={ns_total} in header")
print(f"byte budget : header = 256 x (ns+1) = 256 x {ns_total + 1} = {header_bytes:,} B  |  "
      f"data record = {record_bytes:,} B  x {n_records} records  ({total_bytes:,} B total)")
print(f"precision   : Parquet float64 round-trip max error = {pq_max_err:.3g} uV (lossless)")
print(f"              EDF int16   round-trip max error = {edf_max_err:.4g} uV  (~ quantization step / 2)")
show_table(edf_sig, caption="EDF signal headers (8 EEG signals) — each channel quantized to 16-bit integers")
# === Figure 3 — EDF physical layout (header + interleaved data records) ======================
fig, ax, top = canvas(
    "EDF file anatomy — row-record, header-described",
    "To read one channel: parse the header, then stride through EVERY data record skipping the other signals.")

x0, x1 = 4.0, 96.0
yb, h = 26.0, 13.0

hdr_w = 9.0
sighdr_w = 12.0
rec_area_x = x0 + hdr_w + sighdr_w + 1.5
rec_area_w = x1 - rec_area_x

# header blocks
rect(ax, x0, yb, hdr_w, h, PAL["meta_bg"], PAL["meta"], lw=2.2)
label(ax, x0+hdr_w/2, yb+h-2.6, "Main", size=11, ha="center", weight="bold", color=PAL["meta"])
label(ax, x0+hdr_w/2, yb+h-5.4, "header", size=11, ha="center", weight="bold", color=PAL["meta"])
label(ax, x0+hdr_w/2, yb+3.0, "256 B", size=9, ha="center", color=PAL["muted"])

rect(ax, x0+hdr_w, yb, sighdr_w, h, PAL["meta_bg"], PAL["meta"], lw=2.2)
label(ax, x0+hdr_w+sighdr_w/2, yb+h-2.6, f"{ns_total} signal", size=11, ha="center", weight="bold", color=PAL["meta"])
label(ax, x0+hdr_w+sighdr_w/2, yb+h-5.4, "headers", size=11, ha="center", weight="bold", color=PAL["meta"])
label(ax, x0+hdr_w+sighdr_w/2, yb+3.0, f"{ns_total} x 256 B", size=9, ha="center", color=PAL["muted"])

# a few data records, each split into per-signal blocks
n_show = 4
rec_w = rec_area_w / (n_show + 0.6)
sub_h = (h - 2.0) / ns
for rci in range(n_show):
    rx = rec_area_x + rci * rec_w
    rect(ax, rx, yb, rec_w-1.2, h, PAL["edf_bg"], PAL["edf"], lw=1.8)
    for s in range(ns):
        sy = yb + 1.0 + s * sub_h
        is_t = (s == 0)
        rect(ax, rx+0.8, sy, rec_w-2.8, sub_h-0.5,
             "#f4c66a" if is_t else PAL["panel"], PAL["edf"], lw=0.8, z=4)
    label(ax, rx + (rec_w-1.2)/2, yb-2.4, f"record {rci+1}", size=9.5, ha="center", color=PAL["muted"])
label(ax, rec_area_x + rec_area_w - rec_w*0.6, yb+h/2, "...", size=18, ha="center", color=PAL["edf"])
label(ax, rec_area_x, yb+h+3.2, f"{n_drec} data records (each = sig1 | sig2 | ... | sig{ns} | annot, interleaved)",
      size=12, weight="bold", color=PAL["edf"])

# strided single-channel read path across highlighted sub-block in each record
ys = yb + 1.0 + sub_h/2
pts = [rec_area_x + rci*rec_w + (rec_w-1.2)/2 for rci in range(n_show)]
for k in range(len(pts)-1):
    arrow(ax, (pts[k], ys), (pts[k+1], ys), PAL["ink"], lw=1.8, rad=0.55, z=7)
label(ax, 50, top-12, f"Read '{labels[0]}':  N seeks, one per record (the gold sub-blocks) — "
                      f"the file's full extent is traversed", size=12.5, ha="center", color=PAL["ink"])
label(ax, 50, 9.0, "16-bit integers => fixed precision. Measured round-trip error here: "
                   f"{edf_max_err:.3g} uV vs Parquet's {pq_max_err:.1g} uV.", size=11.5, ha="center", color=PAL["muted"])
save_fig(fig, "fig3_edf_layout")
plt.show(); plt.close(fig)


### Scaling: where the cost reverses

Small recordings can *favour EDF* — a compact fixed header beats Parquet's richer footer and per-column metadata at
low sample counts. The picture **reverses with scale**: reading one channel from EDF strides the whole interleaved
file (cost grows with `channels x samples`), while Parquet reads a single column chunk (cost grows with `samples`).
Below we sweep recording size on randomly generated data and watch the crossover.


In [ ]:
# === Figure 4 — the decisive contrast: read ONE channel ======================================
fig, ax, top = canvas("Reading one channel: contiguous column vs strided records",
                      "Same data, same query — the access pattern is set entirely by the on-disk layout.")
tp = top

# left: Parquet
lx = 6
rect(ax, lx, 30, 40, 16, PAL["parquet_bg"], PAL["parquet"], lw=2.4)
label(ax, lx+20, 44.5, "Parquet", size=16, weight="bold", ha="center", color=PAL["parquet"])
for i in range(8):
    fc = PAL["parquet"] if i == 0 else PAL["panel"]
    rect(ax, lx+3+i*4.4, 33, 3.6, 9, fc, PAL["parquet"], lw=1.2, z=4)
arrow(ax, (lx+4.8, 28), (lx+4.8, 32.6), PAL["ink"], lw=2.6)
label(ax, lx+20, 25.5, "1 seek -> 1 contiguous chunk", size=12.5, ha="center", weight="bold")
label(ax, lx+20, 21.5, "read 1 of N columns", size=11.5, ha="center", color=PAL["muted"])

# right: EDF
rx = 56
rect(ax, rx, 30, 40, 16, PAL["edf_bg"], PAL["edf"], lw=2.4)
label(ax, rx+20, 44.5, "EDF", size=16, weight="bold", ha="center", color=PAL["edf"])
for rci in range(8):
    bx = rx+3+rci*4.4
    for s in range(4):
        fc = "#f4c66a" if s == 0 else PAL["panel"]
        rect(ax, bx, 33+s*2.1, 3.6, 1.8, fc, PAL["edf"], lw=0.7, z=4)
    arrow(ax, (bx+1.8, 31.4), (bx+1.8, 32.8), PAL["ink"], lw=1.4, z=6)
label(ax, rx+20, 25.5, "N seeks -> 1 per record", size=12.5, ha="center", weight="bold")
label(ax, rx+20, 21.5, "skip the other channels each time", size=11.5, ha="center", color=PAL["muted"])

label(ax, 50, 13.5, "Consequence (paper, Exp. 3): single-channel read latency", size=13, ha="center", weight="bold")
label(ax, 50, 9.0, "SBIDS+Parquet ~1.7 ms   vs   EDF ~10 ms   =>   5.88x faster", size=15, ha="center", color=PAL["parquet"], weight="bold")
save_fig(fig, "fig4_read_contrast")
plt.show(); plt.close(fig)
# === Signal read-scaling sweep: read one channel as the recording grows (random data) ========
SWEEP_CH = 64
sweep_dir = ARTIFACT_DIR / "scaling"
shutil.rmtree(sweep_dir, ignore_errors=True); sweep_dir.mkdir(parents=True, exist_ok=True)

srows = []
for d in SWEEP_DURS:
    sd = CortiDataset.generate_eeg_samples(
        sampling_rate=250.0, duration_s=d,
        channel_names=[f"Ch{i+1:02d}" for i in range(SWEEP_CH)], channel_count=SWEEP_CH,
        seed=int(d * 7) + 3, dataset_name="scale")
    pqf, edff = sweep_dir / f"s_{d}.parquet", sweep_dir / f"s_{d}.edf"
    pq.write_table(pa.Table.from_pandas(pd.DataFrame(sd.data, columns=sd.raw.ch_names)),
                   pqf, row_group_size=5000, compression="snappy")
    BIDSLoader(sweep_dir)._write_raw(sd.raw, edff, format="edf", overwrite=True)
    tgt = sd.raw.ch_names[SWEEP_CH // 2]
    t_pq = median_ms(lambda: pq.read_table(pqf, columns=[tgt]).to_pandas(), n=7)
    t_edf = median_ms(lambda: mne.io.read_raw_edf(edff, preload=False, verbose="ERROR").get_data(picks=[tgt]), n=7)
    srows.append({"seconds": d, "samples": int(sd.data.shape[0] * SWEEP_CH),
                  "Parquet ms": t_pq, "EDF ms": t_edf,
                  "Parquet KB": pqf.stat().st_size / 1024, "EDF KB": edff.stat().st_size / 1024})
    del sd
sweep = pd.DataFrame(srows).sort_values("samples").reset_index(drop=True)
sweep["EDF/Parquet"] = sweep["EDF ms"] / sweep["Parquet ms"]

sp, sz = sweep["EDF/Parquet"].to_numpy(), sweep["samples"].to_numpy()
cross = None
for i in range(len(sp) - 1):
    if (sp[i] - 1) * (sp[i + 1] - 1) < 0:
        cross = int(round(np.sqrt(sz[i] * sz[i + 1]))); break
verdict = (f"EDF leads when small, Parquet leads at scale — crossover near {cross:,} samples"
           if cross else
           f"Parquet leads across this range; its margin widens {sp[0]:.1f}x -> {sp[-1]:.1f}x with size")

fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0))
a = axes[0]
a.plot(sweep["samples"], sweep["Parquet ms"], "o-", color=PAL["parquet"], lw=2.6, label="Parquet — 1 column")
a.plot(sweep["samples"], sweep["EDF ms"], "s-", color=PAL["edf"], lw=2.6, label="EDF — 1 channel (MNE)")
a.set_xscale("log"); a.set_yscale("log")
a.set_xlabel("recording size — total samples (log)"); a.set_ylabel("median read time, ms (log)")
a.set_title("Read one channel", fontsize=15); a.legend(); a.grid(alpha=0.3, which="both")
b = axes[1]
b.plot(sweep["samples"], sweep["EDF/Parquet"], "o-", color=PAL["meta"], lw=2.8)
b.axhline(1.0, color=PAL["muted"], ls="--", lw=1.5)
b.fill_between(sweep["samples"], 1.0, sweep["EDF/Parquet"], where=sweep["EDF/Parquet"] >= 1, color=PAL["parquet"], alpha=0.15)
b.fill_between(sweep["samples"], sweep["EDF/Parquet"], 1.0, where=sweep["EDF/Parquet"] < 1, color=PAL["edf"], alpha=0.18)
b.set_xscale("log"); b.set_xlabel("recording size — total samples (log)")
b.set_ylabel("EDF / Parquet   ( >1 : Parquet faster )"); b.set_title("The reversal", fontsize=15); b.grid(alpha=0.3, which="both")
fig.suptitle("Read-one-channel scaling on randomly generated data", fontsize=18, fontweight="bold")
fig.text(0.5, -0.04, verdict + ".  Mechanism: Parquet reads 1 of 64 columns; EDF strides the whole interleaved file.",
         ha="center", fontsize=11.5, color=PAL["muted"])
fig.tight_layout(rect=[0, 0, 1, 0.93])
save_fig(fig, "fig5_read_scaling")
plt.show(); plt.close(fig)
show_table(sweep.round(2), caption="Signal read-scaling sweep (64 ch): EDF's fixed-header edge erodes as size grows")


## Part C — Surgical: the metadata layer (BIDS vs SBIDS / JSON-LD)

The signal layer answers *"how are the numbers stored."* The metadata layer answers *"how is their meaning stored,"*
and how cheaply a machine can recover it.


In [ ]:
# === BIDS metadata introspection: meaning dispersed across the folder + filename convention ==
def tree_files(root):
    return sorted(p for p in root.rglob("*") if p.is_file())

bids_files = tree_files(parquet_root)
contrib = {
    "dataset_description.json": "dataset name (root sidecar)",
    "_eeg.json": "SamplingFrequency (per-run sidecar)",
    "_channels.tsv": "channel names + types (per-run table)",
    "_eeg.parquet": "the raw signal",
    "_eeg.edf": "the raw signal",
    "_events.tsv": "events",
}
def role(name):
    for k, v in contrib.items():
        if name.endswith(k):
            return v
    return ""

print(f"BIDS dataset tree under {parquet_root.as_posix()}/\n")
for p in bids_files:
    rel = p.relative_to(parquet_root)
    depth = len(rel.parts) - 1
    print(f"  {'  '*depth}{rel.parts[-1]:<48s} <- {role(p.name)}")

# the same fact, two layouts: where does 'sampling rate of sub-01 task-formatdemo' live?
sidecar = parquet_path.with_suffix(".json")
channels_tsv = parquet_path.with_name(parquet_path.stem.rsplit('_', 1)[0] + "_channels.tsv")
print("\nQuery: 'sampling rate + channels for sub-01, task formatdemo?'  In BIDS you must:")
print(f"  1. crawl directories to sub-01/ses-demo/eeg/")
print(f"  2. parse filename entities  (sub-01_ses-demo_task-formatdemo_run-01)")
print(f"  3. open {sidecar.name}  -> {json.loads(sidecar.read_text())}")
print(f"  4. open {channels_tsv.name}  -> {pd.read_csv(channels_tsv, sep=chr(9))['name'].tolist()}")
n_bids_meta = sum(1 for p in bids_files if p.suffix.lower() in {".json", ".tsv"})
print(f"\n=> meaning is spread over {n_bids_meta} sidecar files + the path/filename convention.")
# === Figure 6 — BIDS metadata layout (folder hierarchy + scattered sidecars) =================
fig, ax, top = canvas("BIDS: meaning lives in the folder/filename convention + sidecars",
                      "A consumer recovers context by crawling paths, parsing entities, and opening several sidecar files.")
nodes = [
    (6,  44, "dataset_root/", PAL["panel"], 0, "root"),
    (10, 39, "dataset_description.json", PAL["bids_bg"], 1, "dataset name"),
    (10, 34, "sub-01/", PAL["panel"], 1, "subject (in path)"),
    (14, 29, "ses-demo/", PAL["panel"], 2, "session (in path)"),
    (18, 24, "eeg/", PAL["panel"], 3, "modality (in path)"),
    (22, 19, "..._eeg.json", PAL["bids_bg"], 4, "SamplingFrequency"),
    (22, 14, "..._channels.tsv", PAL["bids_bg"], 4, "channel names + types"),
    (22, 9,  "..._eeg.parquet", PAL["parquet_bg"], 4, "the raw signal"),
]
for x, y, name, fc, depth, note in nodes:
    w = 30
    rect(ax, x, y, w, 4.0, fc, PAL["bids"] if fc != PAL["panel"] else PAL["edge"], lw=1.8)
    label(ax, x+1.6, y+2.0, name, size=12, family="monospace",
          weight="bold" if fc != PAL["panel"] else "normal")
    arrow(ax, (x+w+1, y+2.0), (62, y+2.0), PAL["muted"], lw=1.2, style="-", rad=0)
    label(ax, 63, y+2.0, note, size=11.5, color=PAL["muted"])
label(ax, 6, 5.2, "Identity (sub/ses/task/run) is encoded in strings; a parser, not the data, supplies the semantics.",
      size=12, color=PAL["ink"])
save_fig(fig, "fig6_bids_layout")
plt.show(); plt.close(fig)


In [ ]:
# === SBIDS introspection: one JSON-LD knowledge graph, typed nodes, explicit links ===========
sbids_doc = json.loads(Path(sbids_path).read_text(encoding="utf-8"))
graph = sbids_doc["@graph"]

def typ(n):
    t = n.get("@type", "")
    return " + ".join(t) if isinstance(t, list) else t

node_rows = []
for n in graph:
    node_rows.append({"@id": n.get("@id", ""), "@type": typ(n),
                      "name": n.get("schema:name", n.get("schema:identifier", ""))})
sbids_nodes = pd.DataFrame(node_rows)

rec = next(n for n in graph if "schema:CreateAction" in (n.get("@type") or []))
file_node = next(n for n in graph if "schema:DigitalDocument" in (n.get("@type") or []))
n_channels = len(rec.get("schema:variableMeasured", []))

print("@context prefixes :", ", ".join(list(sbids_doc["@context"].keys())[:8]), "...")
print(f"graph nodes       : {len(graph)}  (+ {n_channels} channel sub-nodes inside the recording)")
print(f"recording links   : instrument -> {rec['schema:instrument']['@id']}")
print(f"                    object     -> {rec['schema:object']['@id']}")
print(f"                    result     -> {rec['schema:result']['@id']}")
print(f"file node         : contentUrl -> {file_node['schema:contentUrl']}  ({file_node['schema:encodingFormat']})")

subj_id = rec["schema:object"]["@id"].split(":")[-1]
sparql = "\n".join([
    "SELECT ?file ?fs WHERE {",
    "  ?rec  a schema:CreateAction ;",
    "        schema:object  ?subj ;",
    "        schema:result  ?file ;",
    '        schema:additionalProperty [ schema:name "SamplingRate" ; schema:value ?fs ] .',
    '  ?subj schema:identifier "%s" .' % subj_id,
    "  ?file schema:contentUrl ?url .",
    "}",
])
print("\nSame query as before, as one graph traversal (SPARQL-style):\n")
print(sparql)
print("\n=> one file, one traversal; no path-crawling, no filename parsing.")
show_table(sbids_nodes, caption="SBIDS JSON-LD @graph — typed nodes linked into one queryable document")
# === Figure 7 — SBIDS knowledge graph (drawn from the real node ids) =========================
fig, ax, top = canvas("SBIDS: meaning is one linked JSON-LD graph",
                      "Typed nodes and explicit edges; the file node points to the raw signal via contentUrl.")

def gnode(x, y, title, sub, fc, ec, w=22, h=8):
    rect(ax, x, y, w, h, fc, ec, lw=2.3)
    label(ax, x+w/2, y+h-2.6, title, size=13.5, weight="bold", ha="center", color=ec)
    label(ax, x+w/2, y+2.6, sub, size=10.5, ha="center", color=PAL["muted"], family="monospace")
    return (x, y, w, h)

ds = gnode(39, 40, "Dataset", "schema:Dataset", PAL["panel"], PAL["ink"])
rc = gnode(39, 28, "Recording", "CreateAction + prov:Activity", PAL["sbids_bg"], PAL["sbids"], w=24)
su = gnode(8, 28, "Subject", "schema:Patient", PAL["meta_bg"], PAL["meta"])
dv = gnode(8, 14, "Device", "MedicalDevice", PAL["meta_bg"], PAL["meta"])
fl = gnode(70, 28, "File", "DigitalDocument", PAL["panel"], PAL["ink"])
pq_n = gnode(70, 13, "raw_data/*.parquet", "encodingFormat: parquet", PAL["parquet_bg"], PAL["parquet"], w=26)

# dataset -> recording
arrow(ax, (50, 40), (50, 36.2), PAL["ink"], rad=0); label(ax, 52.5, 38.0, "hasRecording", size=10.5, color=PAL["muted"])
# recording -> subject (object)
arrow(ax, (39, 31.5), (30.3, 31.5), PAL["sbids"], rad=0); label(ax, 34.7, 33.6, "object", size=10.5, color=PAL["sbids"], ha="center")
# recording -> device (instrument)
arrow(ax, (39, 29.0), (17, 22.2), PAL["sbids"], rad=0.12); label(ax, 31, 26.2, "instrument", size=10.5, color=PAL["sbids"], ha="center")
# recording -> file (result)
arrow(ax, (63, 31.5), (70, 31.5), PAL["sbids"], rad=0); label(ax, 66.5, 33.6, "result", size=10.5, color=PAL["sbids"], ha="center")
# file -> parquet (contentUrl)
arrow(ax, (83, 28), (83, 21), PAL["parquet"], rad=0); label(ax, 85.5, 24.5, "contentUrl", size=10.5, color=PAL["parquet"], ha="center")
# file -> recording (prov:wasGeneratedBy) back-edge
arrow(ax, (70, 30), (63, 30), PAL["muted"], rad=0.45, ls="--", lw=1.4)

# channel chips under recording
label(ax, 51, 25.2, f"variableMeasured: {n_channels} EEGChannel nodes", size=10.5, ha="center", color=PAL["muted"])
for i, ch in enumerate(CH):
    cx = 33 + (i % 8) * 4.6
    rect(ax, cx, 20.5, 4.0, 3.0, PAL["sbids_bg"], PAL["sbids"], lw=1.0, z=4)
    label(ax, cx+2.0, 22.0, ch, size=8.0, ha="center", color=PAL["sbids"])

label(ax, 50, 7.5, "One question -> one graph query. JSON-LD adds structural flexibility without losing semantic comparability.",
      size=12, ha="center", color=PAL["muted"])
save_fig(fig, "fig7_sbids_graph")
plt.show(); plt.close(fig)
# === Figure 8 — BIDS vs SBIDS: cost of answering one question ================================
fig, ax, top = canvas("Recovering meaning: crawl vs query",
                      "Same answer ('which file + sampling rate for sub-01?'); very different machine effort.")
# left BIDS
lx = 6
rect(ax, lx, 14, 40, 30, PAL["bids_bg"], PAL["bids"], lw=2.4)
label(ax, lx+20, 41, "BIDS", size=17, weight="bold", ha="center", color=PAL["bids"])
steps = ["1.  crawl directory tree", "2.  parse filename entities",
         "3.  open *_eeg.json (sfreq)", "4.  open *_channels.tsv (chans)"]
for i, s in enumerate(steps):
    label(ax, lx+3, 36-i*5.2, s, size=12.5, family="monospace")
label(ax, lx+20, 16.5, f"{n_bids_meta} sidecars + convention", size=12, ha="center", color=PAL["bids"], weight="bold")

# right SBIDS
rx = 56
rect(ax, rx, 14, 40, 30, PAL["sbids_bg"], PAL["sbids"], lw=2.4)
label(ax, rx+20, 41, "SBIDS", size=17, weight="bold", ha="center", color=PAL["sbids"])
label(ax, rx+3, 36, "1.  one SPARQL-style query", size=12.5, family="monospace")
label(ax, rx+3, 30.8, "    over one JSON-LD graph", size=12.5, family="monospace", color=PAL["muted"])
label(ax, rx+3, 24, f"{len(graph)} nodes, explicit links", size=12.5)
label(ax, rx+20, 16.5, "1 file, 1 traversal", size=12, ha="center", color=PAL["sbids"], weight="bold")

label(ax, 50, 8.5, "Paper (Exp. 3): metadata-layer overhead (JSON-LD vs sidecars) is negligible; "
                   "the 5.88x read win comes from the Parquet raw layout.",
      size=11.8, ha="center", color=PAL["muted"])
save_fig(fig, "fig8_metadata_contrast")
plt.show(); plt.close(fig)


### Containers side by side: redundancy and on-disk footprint

The single-recording view above isolates the *mechanism*. Redundancy shows up at **cohort scale** — here
2 subjects x 2 runs. BIDS repeats each run's sidecars on disk; SBIDS keeps shared entities once in one graph.
Below: the two containers side by side, a post-retrieval table exposing the duplicated metadata, and an
**honest** size comparison (lossless-vs-lossy and file-count-vs-bytes — not a free win).


In [ ]:
# === Build a 4-recording cohort (2 subjects x 2 runs) so BIDS redundancy is concrete =========
multi_bids = ARTIFACT_DIR / "bids_multi"
sbids_multi_dir = ARTIFACT_DIR / "sbids_multi"
shutil.rmtree(multi_bids, ignore_errors=True)
shutil.rmtree(sbids_multi_dir, ignore_errors=True)

cohort = [("01", "01"), ("01", "02"), ("02", "01"), ("02", "02")]  # (subject, run) — shared montage
for subj, run in cohort:
    ds_i = CortiDataset.generate_eeg_samples(
        sampling_rate=FS, duration_s=10.0, channel_names=CH, channel_count=len(CH),
        seed=100 + int(subj) * 10 + int(run), dataset_name=f"sub{subj}run{run}")
    ds_i.to_bids(multi_bids, subject=subj, session="01", task="rest", run=run,
                 format="parquet", overwrite=True, dataset_description={"Name": "CortiPy cohort demo"})

sbids_multi = to_sbids(multi_bids, all_recordings=True,
                       output=sbids_multi_dir / "sbids_meta_cohort.jsonld", export_format="parquet")

n_bids_meta_multi = sum(1 for p in multi_bids.rglob("*") if p.suffix.lower() in {".json", ".tsv"})
doc_multi = json.loads(Path(sbids_multi).read_text(encoding="utf-8"))
g_multi = doc_multi["@graph"]
n_graph_multi = len(g_multi)
n_subj_multi = sum(1 for n in g_multi if n.get("@type") == "schema:Patient")
n_dev_multi = sum(1 for n in g_multi if "schema:MedicalDevice" in (n.get("@type") or []))

print(f"cohort: {len(cohort)} recordings (2 subjects x 2 runs), shared montage = {CH}\n")
print(f"BIDS container   {multi_bids.as_posix()}/")
for p in sorted(multi_bids.rglob("*")):
    if p.is_file():
        print("   " + p.relative_to(multi_bids).as_posix())
print(f"\nSBIDS container  {sbids_multi_dir.as_posix()}/")
for p in sorted(sbids_multi_dir.rglob("*")):
    if p.is_file():
        print("   " + p.relative_to(sbids_multi_dir).as_posix())
print(f"\nBIDS metadata files: {n_bids_meta_multi} (2N+1)   |   SBIDS metadata files: 1 "
      f"({n_graph_multi}-node graph; Subject x{n_subj_multi}, Device x{n_dev_multi} stored once)")
# === Figure 9 — BIDS vs SBIDS containers, side by side =======================================
fig, ax, top = canvas("BIDS vs SBIDS containers",
                      "Same 4 recordings. Left: metadata scattered across many files. Right: one graph + raw_data/.")

# left: BIDS
lx, lw_ = 4, 44
rect(ax, lx, 9, lw_, 36, PAL["bids_bg"], PAL["bids"], lw=2.4)
label(ax, lx + 2, 42.5, "BIDS  —  bids_multi/", size=14.5, weight="bold", color=PAL["bids"])
bids_lines = [
    ("dataset_description.json", PAL["bids"]),
    ("sub-01/ses-01/eeg/", PAL["ink"]),
    ("   run-01:  _eeg.parquet | _eeg.json | _channels.tsv", PAL["muted"]),
    ("   run-02:  _eeg.parquet | _eeg.json | _channels.tsv", PAL["muted"]),
    ("sub-02/ses-01/eeg/", PAL["ink"]),
    ("   run-01:  _eeg.parquet | _eeg.json | _channels.tsv", PAL["muted"]),
    ("   run-02:  _eeg.parquet | _eeg.json | _channels.tsv", PAL["muted"]),
]
y = 39
for txt, col in bids_lines:
    label(ax, lx + 2, y, txt, size=9.5, family="monospace", color=col)
    y -= 3.2
label(ax, lx + 2, 14.0, f"{n_bids_meta_multi} metadata files (2N+1).", size=11.5, weight="bold", color=PAL["bids"])
label(ax, lx + 2, 11.0, "_channels.tsv byte-identical across all 4 runs.", size=10, color=PAL["muted"])

# right: SBIDS
rx, rw_ = 52, 44
rect(ax, rx, 9, rw_, 36, PAL["sbids_bg"], PAL["sbids"], lw=2.4)
label(ax, rx + 2, 42.5, "SBIDS  —  sbids_multi/", size=14.5, weight="bold", color=PAL["sbids"])
sb_lines = [
    ("sbids_meta_cohort.jsonld", PAL["sbids"]),
    (f"   one graph: {n_graph_multi} nodes", PAL["muted"]),
    ("raw_data/", PAL["ink"]),
    ("   sub-01_..._run-01_eeg.parquet", PAL["parquet"]),
    ("   sub-01_..._run-02_eeg.parquet", PAL["parquet"]),
    ("   sub-02_..._run-01_eeg.parquet", PAL["parquet"]),
    ("   sub-02_..._run-02_eeg.parquet", PAL["parquet"]),
]
y = 39
for txt, col in sb_lines:
    label(ax, rx + 2, y, txt, size=9.5, family="monospace", color=col)
    y -= 3.2
label(ax, rx + 2, 14.0, "1 metadata file.", size=11.5, weight="bold", color=PAL["sbids"])
label(ax, rx + 2, 11.0, f"Subject x{n_subj_multi} & Device x{n_dev_multi} stored once, linked by URN.",
      size=10, color=PAL["muted"])

arrow(ax, (lx + lw_ + 0.3, 27), (rx - 0.3, 27), PAL["ink"], lw=2.0)
save_fig(fig, "fig9_containers_side_by_side")
plt.show(); plt.close(fig)


In [ ]:
# === Table after retrieval: the same facts, physically duplicated across BIDS sidecars ========
loader_multi = BIDSLoader(multi_bids)
results_multi = loader_multi.read_bids_dataset(allowed_file_structures=(".parquet",))

rows = []
for res in results_multi:
    src = Path(res.source_path)
    base = src.stem.rsplit("_", 1)[0]                 # sub-01_ses-01_task-rest_run-01
    sidecar = src.with_suffix(".json")
    chtsv = src.with_name(base + "_channels.tsv")
    rows.append({
        "recording": base,
        "fs (Hz)": json.loads(sidecar.read_text())["SamplingFrequency"],
        "n ch": int(res.data.shape[1]),
        "units": "uV",
        "channels.tsv sha1": hashlib.sha1(chtsv.read_bytes()).hexdigest()[:10],
        "physically stored in": f"{sidecar.name}  +  {chtsv.name}",
    })
redundancy = pd.DataFrame(rows)
uniq = redundancy["channels.tsv sha1"].nunique()
print(f"Retrieved {len(results_multi)} recordings. fs / channel count / units / the channel table are identical,")
print(f"yet each pair lives in its own files: the channels.tsv montage has {uniq} unique content hash but is")
print(f"physically stored {len(results_multi)} times. SBIDS keeps shared Subject/Device entities once in the graph.")
show_table(redundancy,
           caption="After retrieval from BIDS: identical metadata duplicated across per-run sidecars (note the repeated sha1)")


In [ ]:
# === File & directory size comparison (honest: lossless-vs-lossy, file-count-vs-bytes) =======
def dir_stats(root):
    files = [p for p in Path(root).rglob("*") if p.is_file()]
    meta = sum(p.stat().st_size for p in files if p.suffix.lower() in {".json", ".tsv", ".jsonld"})
    total = sum(p.stat().st_size for p in files)
    return len(files), meta, total - meta, total

b_files, b_meta, b_raw, b_total = dir_stats(multi_bids)
s_files, s_meta, s_raw, s_total = dir_stats(sbids_multi_dir)

size_tbl = pd.DataFrame([
    {"container": "BIDS  (dir)", "files": b_files, "metadata files": n_bids_meta_multi,
     "metadata B": b_meta, "raw B": b_raw, "total B": b_total},
    {"container": "SBIDS (dir)", "files": s_files, "metadata files": 1,
     "metadata B": s_meta, "raw B": s_raw, "total B": s_total},
])

pq_bytes, edf_bytes = parquet_path.stat().st_size, edf_path.stat().st_size

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.6))
axes[0].bar(["Parquet\n(float64)", "EDF\n(int16)"], [pq_bytes/1024, edf_bytes/1024],
            color=[PAL["parquet"], PAL["edf"]], edgecolor="white")
axes[0].set_title("Raw file, one recording", fontsize=14); axes[0].set_ylabel("KB")
axes[0].text(0, pq_bytes/1024, "lossless", ha="center", va="bottom", fontsize=11, color=PAL["parquet"])
axes[0].text(1, edf_bytes/1024, "16-bit lossy", ha="center", va="bottom", fontsize=11, color=PAL["edf"])

axes[1].bar(["BIDS", "SBIDS"], [n_bids_meta_multi, 1], color=[PAL["bids"], PAL["sbids"]], edgecolor="white")
axes[1].set_title("Metadata files (cohort)", fontsize=14); axes[1].set_ylabel("# files")
for i, v in enumerate([n_bids_meta_multi, 1]):
    axes[1].text(i, v, str(v), ha="center", va="bottom", fontsize=12)

axes[2].bar(["BIDS", "SBIDS"], [b_meta/1024, s_meta/1024], color=[PAL["bids"], PAL["sbids"]], edgecolor="white")
axes[2].set_title("Metadata bytes (cohort)", fontsize=14); axes[2].set_ylabel("KB")
for i, v in enumerate([b_meta/1024, s_meta/1024]):
    axes[2].text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=11)

for ax_ in axes:
    ax_.grid(axis="y", alpha=0.3)
fig.suptitle("On-disk footprint — a tradeoff, not a free win", fontsize=18, fontweight="bold")
fig.text(0.5, -0.04, "EDF is smaller but quantized; Parquet is larger but lossless & column-addressable.  "
                     "SBIDS trades more verbose, self-describing metadata bytes for far fewer files (1 vs 2N+1).",
         ha="center", fontsize=11, color=PAL["muted"])
fig.tight_layout(rect=[0, 0, 1, 0.94])
save_fig(fig, "fig10_container_sizes")
plt.show(); plt.close(fig)
show_table(size_tbl,
           caption="Directory footprint: BIDS = many small sidecars; SBIDS = one richer graph (more metadata bytes, far fewer files; raw signal bytes dominate the total either way)")
# === Metadata-scaling sweep: resolve one recording in a cohort as N grows =====================
mrows = []
for N in META_NS:
    broot, sroot = ARTIFACT_DIR / f"scale_bids_{N}", ARTIFACT_DIR / f"scale_sbids_{N}"
    shutil.rmtree(broot, ignore_errors=True); shutil.rmtree(sroot, ignore_errors=True)
    for k in range(N):
        ds_k = CortiDataset.generate_eeg_samples(
            sampling_rate=250.0, duration_s=2.0, channel_names=CH, channel_count=len(CH),
            seed=500 + k, dataset_name=f"r{k}")
        ds_k.to_bids(broot, subject=f"{k+1:02d}", session="01", task="rest", run="01",
                     format="parquet", overwrite=True, dataset_description={"Name": f"cohort{N}"})
    sj = Path(to_sbids(broot, all_recordings=True, output=sroot / "meta.jsonld", export_format="parquet"))
    tsub = f"{N:02d}"  # resolve the LAST subject — worst case for a directory crawl

    def bids_resolve():
        ld = BIDSLoader(broot)
        recs, _ = ld._discover_recordings((".parquet",), ("eeg", "ieeg"))
        m = next(p for p in recs if f"sub-{tsub}_" in p.stem)
        return json.loads(m.with_suffix(".json").read_text())["SamplingFrequency"]

    def sbids_resolve():
        g = json.loads(sj.read_text())["@graph"]
        rec = next(n for n in g if isinstance(n.get("schema:object"), dict)
                   and n["schema:object"]["@id"].endswith(":" + tsub))
        fnode = next(n for n in g if n.get("@id") == rec["schema:result"]["@id"])
        fs = next(p["schema:value"] for p in rec.get("schema:additionalProperty", [])
                  if p.get("schema:name") == "SamplingRate")
        return fnode["schema:contentUrl"], fs

    mrows.append({"N": N, "BIDS files": 2 * N + 1, "SBIDS files": 1,
                  "BIDS ms": median_ms(bids_resolve, n=15), "SBIDS ms": median_ms(sbids_resolve, n=15)})
meta = pd.DataFrame(mrows)

fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0))
a = axes[0]
a.plot(meta["N"], meta["BIDS files"], "o-", color=PAL["bids"], lw=2.6, label="BIDS  (2N+1 files)")
a.plot(meta["N"], meta["SBIDS files"], "s-", color=PAL["sbids"], lw=2.6, label="SBIDS (1 file)")
a.set_xlabel("recordings in cohort (N)"); a.set_ylabel("# metadata files")
a.set_title("Metadata files on disk", fontsize=15); a.legend(); a.grid(alpha=0.3)
b = axes[1]
b.plot(meta["N"], meta["BIDS ms"], "o-", color=PAL["bids"], lw=2.6, label="BIDS — crawl + sidecar")
b.plot(meta["N"], meta["SBIDS ms"], "s-", color=PAL["sbids"], lw=2.6, label="SBIDS — graph query")
b.set_xlabel("recordings in cohort (N)"); b.set_ylabel("resolve one recording, ms")
b.set_title("Targeted retrieval latency", fontsize=15); b.legend(); b.grid(alpha=0.3)
fig.suptitle("Metadata scaling: file sprawl and lookup cost vs cohort size", fontsize=18, fontweight="bold")
fig.text(0.5, -0.04, "BIDS scatters 2N+1 sidecars and crawls them to find one target; SBIDS keeps a single graph. "
                     "BIDS file count and lookup cost grow ~linearly with N while SBIDS stays flat.", ha="center", fontsize=11.5, color=PAL["muted"])
fig.tight_layout(rect=[0, 0, 1, 0.93])
save_fig(fig, "fig11_metadata_scaling")
plt.show(); plt.close(fig)
show_table(meta.round(3), caption="Metadata scaling: BIDS file count and crawl cost grow with N; SBIDS stays at one file")


## Part D — Discussion crib sheet (for the hallway conversation)

**Signal layer — Parquet vs EDF**
- *Parquet* = footer-indexed columnar table. One channel = one **contiguous column chunk**; the footer carries
  schema, byte offsets, and **min/max statistics** → projection + predicate pushdown, native float64 (lossless).
- *EDF* = fixed header + **interleaved data records** of 16-bit integers. One channel = a sub-block repeated in
  every record → a **strided traversal of the whole file**, with quantization fixed by `(phys_max−phys_min)/65535`.
- Net: single-channel read ≈ **5.88× faster** on Parquet (paper, Exp. 3, M2 Max), reproduced in mechanism by Fig. 5.

**Metadata layer — BIDS vs SBIDS**
- *BIDS* = meaning in the **folder/filename convention** + scattered `.json`/`.tsv` sidecars; recovery = crawl + parse + open several files.
- *SBIDS* = one **JSON-LD knowledge graph** of typed nodes (Dataset/Subject/Device/Recording/File) with explicit
  links and `contentUrl` to the raw signal; recovery = one graph query.
- Honest framing: the metadata layer's I/O overhead is **negligible** vs BIDS sidecars — its value is queryability and
  interoperability, *not* the speed number. The speed comes from Parquet.
- At **cohort scale** BIDS writes `2N+1` metadata files (the montage `channels.tsv` is byte-identical per run); SBIDS uses
  one graph with shared Subject/Device nodes. Footprint is a **tradeoff**: EDF smallest-but-lossy, Parquet larger-but-lossless,
  SBIDS far-fewer-files but more verbose (self-describing) metadata bytes — not a free win.

**Scaling (the honest crossover)**
- At *small* sizes EDF/BIDS can win on fixed overhead (compact header, tiny sidecars). The advantage **reverses with scale**:
  Parquet single-channel read cost grows with `samples` (one column), EDF with `channels x samples` (strided); BIDS metadata
  cost grows with `2N+1` files crawled, SBIDS stays at one graph. Figs. 5 and 11 show the crossover on random data.

**If asked "isn't this just Parquet + JSON-LD?"** — yes, deliberately. The contribution is the *pairing* (SBIDS) plus a
verified acquisition→storage pipeline (bitwise SHA-256 + EEGLAB-equivalent outputs).


In [ ]:
# === Manifest, README, and optional cleanup (the LAST cell) ==================================
README_TEXT = """# CortiPy EMBC demo - generated artifacts

This folder contains `EMBC_CortiPy_public_demo.ipynb` and its generated demo artifacts.
Re-running the notebook recreates all data from randomly generated EEG.

## How to run
1. Install deps:  `pip install -e .[ui,bids]`  (numpy, pandas, matplotlib, mne, pyarrow, pyedflib)
2. Open `presentation_artifacts/EMBC_CortiPy_public_demo.ipynb` and **Run All**.
3. Figures render inline and export here to `figures/` as PDF + SVG + PNG@300 + PNG@600.

The notebook creates all data and directories itself. The final cell tidies this folder; set
`CLEAN_ARTIFACTS = False` in the setup cell (or env `CORTIPY_CLEAN=false`) to keep the exported figures.
"""

figs = sorted(FIG_DIR.glob("fig*"))
print(f"generated {len(figs)} figure files in {FIG_DIR.as_posix()} "
      f"(each figure: pdf + svg + png@300 + @2x png@600)")

if CLEAN_ARTIFACTS:
    import gc
    for attempt in range(3):
        gc.collect()  # release lingering reader handles (Windows blocks deletion of open files)
        keep = {"README.md", "EMBC_CortiPy_public_demo.ipynb"}
        leftover = [p for p in ARTIFACT_DIR.iterdir() if p.name not in keep]
        if not leftover:
            break
        for p in leftover:
            try:
                shutil.rmtree(p) if p.is_dir() else p.unlink()
            except OSError:
                pass
        time.sleep(0.2)
    (ARTIFACT_DIR / "README.md").write_text(README_TEXT, encoding="utf-8")
    print()
    print("CLEAN_ARTIFACTS=True -> removed generated files; kept only presentation_artifacts/README.md")
    print("Figures remain visible inline above. Set CLEAN_ARTIFACTS=False to keep the exported image files.")
else:
    (ARTIFACT_DIR / "README.md").write_text(README_TEXT, encoding="utf-8")
    man = pd.DataFrame([{"file": p.name, "KB": round(p.stat().st_size / 1024, 1)} for p in figs])
    print()
    print("CLEAN_ARTIFACTS=False -> keeping exported files.")
    show_table(man, caption="presentation_artifacts/figures/ - conference-ready assets (PDF/SVG/PNG)")
